# 6 · Enhancer redundancy — Step 1 (Spark, cluster)

Pure-Spark (like notebook 1, no numpy/pandas on the driver). Finds the nearest **active** enhancer per gene in each cell line and writes two small tables to S3:

- `s3a://database/enhancer_redundancy_nearest` — nearest active enhancer + coords per (cell_line, gene)
- `s3a://database/enhancer_redundancy_ensmap` — (cell_line, chrom) → ensemble_id

The heavy **Step 2** (recompute the c1-nearest enhancer's 3D distance in c2's model) and all analysis run locally in **notebook 7**, where numpy is available. Pull both tables to `data/whole_chromosomes/`. See `docs/superpowers/specs/2026-06-16-enhancer-redundancy-design.md`.

In [ ]:
%%configure -f
{"executorMemory": "12G", "executorCores": 12, "ttl": "12h", "heartbeatTimeoutInSecond": 43200, "numExecutors": 3}

In [ ]:
from pyspark.sql import Window
import pyspark.sql.functions as F

# Broad "active" ChromHMM state set (matches notebook 1).
ACTIVE_STATES = ['TssA', 'TssAFlnk', 'TxFlnk', 'Tx', 'TxWk',
                 'EnhG', 'EnhG1', 'EnhG2', 'Enh', 'EnhA1', 'EnhA2']

# Per-cell-line result query IDs (same as notebook 1; must retain enh/gene coords).
QUERY_IDS = {
    "GM12878": "a1fc46a9-93f8-424f-b41d-37bfd85d3b94",
    "H1ESC":   "f7bc6dac-6aa3-49e6-a2e5-c2ff27824c81",
    "HFFC6":   "f648f805-c3a9-4cf4-a108-94e6f5fa96c1",
}
USED_PROJECTS = ['whole_all_vs_all_gm12878_fix',
                 'whole_all_vs_all_h1esc_fix',
                 'whole_all_vs_all_hffc6_fix']

NEAREST_OUTPUT = "s3a://database/enhancer_redundancy_nearest"
ENSMAP_OUTPUT  = "s3a://database/enhancer_redundancy_ensmap"

In [ ]:
def read_results(cell_line, query_id):
    return (spark.read.parquet(f"s3a://database/results/{query_id}")
            .withColumn("cell_line", F.lit(cell_line)))

results = None
for cl, qid in QUERY_IDS.items():
    df = read_results(cl, qid)
    results = df if results is None else results.union(df)

results = (results
           .where("avg_dist > 0 AND var_dist > 0")
           .where(F.col('project_id').isin(USED_PROJECTS)))

chromatin_states_df = (spark.read.parquet("s3a://database/chromatin_states")
                       .where(F.col('name').isin(ACTIVE_STATES)))
results.createOrReplaceTempView("results")
chromatin_states_df.createOrReplaceTempView("chromatin_states")

In [ ]:
active_pairs = spark.sql("""
SELECT r.gene_id, r.gene_chr, r.gene_start, r.gene_end, r.gene_strand,
       r.enh_id, r.enh_chr, r.enh_start, r.enh_end,
       r.avg_dist, r.cell_line
FROM results r
WHERE EXISTS (SELECT 1 FROM chromatin_states cs
              WHERE cs.cell_line = r.cell_line AND cs.chrom = r.gene_chr
                AND cs.start <= r.gene_end AND cs.end >= r.gene_start)
  AND EXISTS (SELECT 1 FROM chromatin_states cs
              WHERE cs.cell_line = r.cell_line AND cs.chrom = r.enh_chr
                AND cs.start <= r.enh_end AND cs.end >= r.enh_start)
""")

In [ ]:
w = Window.partitionBy('cell_line', 'gene_id').orderBy(F.col('avg_dist').asc())
nearest = (active_pairs
           .withColumn('rk', F.row_number().over(w))
           .where('rk = 1')
           .drop('rk'))

nearest.write.mode('overwrite').parquet(NEAREST_OUTPUT)
print("wrote nearest-active-enhancer table ->", NEAREST_OUTPUT)
nearest.groupBy('cell_line').count().show()

In [ ]:
# (cell_line, chrom) -> ensemble_id, so notebook 7 knows which c2 model to load
proj = (spark.read.json("s3a://database/project_configuration", multiLine=True)
        .select(F.col('project_id'), F.explode('datasets').alias('d'))
        .select(F.col('d.metadata.cell_line').alias('cell_line'),
                F.col('d.ensemble_region.chromosome').alias('chrom'),
                F.col('d.ensemble_id').alias('ensemble_id'),
                F.col('project_id'))
        .where(F.col('project_id').isin(USED_PROJECTS))
        .select('cell_line', 'chrom', 'ensemble_id').distinct())

proj.write.mode('overwrite').parquet(ENSMAP_OUTPUT)
print("wrote ensemble map ->", ENSMAP_OUTPUT)
proj.show(10)